# 11_score_base_difficulty_colab

v6A 第二步：先让 Qwen3-ASR base 在 v6A hard-profile train/val 上跑推理和 WER，再生成 difficulty manifest。这个 notebook 不训练 LoRA。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# 固定 Qwen3-ASR 已验证依赖组合。bitsandbytes 用于 4bit base，对齐 LoRA 评测口径。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr==0.0.6 transformers==4.57.6 accelerate==1.12.0 bitsandbytes huggingface_hub pyyaml
%pip -q install pandas==2.2.2 requests==2.32.4

In [ ]:
from pathlib import Path
from collections import Counter
import json
import os
import subprocess
import sys
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/eval/qwen3_asr_v6a_base_difficulty.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

MODEL_ID = config['model'].get('id', 'Qwen/Qwen3-ASR-1.7B')
DTYPE = config['model'].get('dtype', 'float16')
DEVICE_MAP = config['model'].get('device_map', 'cuda:0')
QUANTIZATION = config['model'].get('quantization', '4bit')
MAX_INFERENCE_BATCH_SIZE = int(config['model'].get('max_inference_batch_size', 1))
MAX_NEW_TOKENS = int(config['inference'].get('max_new_tokens', 128))
LANGUAGE = config['inference'].get('language', 'English')
AUDIO_ROOT = PROJECT_DIR
LIMIT = int(config.get('runtime', {}).get('limit', 0) or 0)
MINI_LIMIT_PER_SPLIT = int(config.get('runtime', {}).get('mini_limit_per_split', 7) or 7)

TRAIN_MANIFEST = PROJECT_DIR / config['input']['train_manifest']
VAL_MANIFEST = PROJECT_DIR / config['input']['val_manifest']
OUTPUT_DIR = PROJECT_DIR / config['output']['dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIFFICULTY = PROJECT_DIR / config['output']['train_difficulty_manifest']
VAL_DIFFICULTY = PROJECT_DIR / config['output']['val_difficulty_manifest']
SUMMARY_JSON = PROJECT_DIR / config['output']['summary_json']

print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG =', CONFIG_PATH)
print('MODEL_ID =', MODEL_ID, 'quantization =', QUANTIZATION)
print('TRAIN_MANIFEST =', TRAIN_MANIFEST)
print('VAL_MANIFEST =', VAL_MANIFEST)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# 检查 GPU、脚本、manifest 和音频路径。
import torch

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)
else:
    raise RuntimeError('当前 runtime 没有 CUDA GPU，无法跑 Qwen3-ASR base scoring。')

required = [
    CONFIG_PATH,
    PROJECT_DIR / 'inference/qwen3_asr_base_infer.py',
    PROJECT_DIR / 'evaluation/eval_wer.py',
    PROJECT_DIR / 'evaluation/analyze_errors.py',
    PROJECT_DIR / 'scripts/build_difficulty_manifest.py',
    TRAIN_MANIFEST,
    VAL_MANIFEST,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('缺少必要文件:\n' + '\n'.join(missing))

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def resolve_audio(audio):
    path = Path(audio)
    if path.is_absolute():
        return path
    return PROJECT_DIR / path

for manifest, expected_rows in [(TRAIN_MANIFEST, 1680), (VAL_MANIFEST, 420)]:
    rows = read_jsonl(manifest)
    counts = Counter(row.get('scenario', '') for row in rows)
    missing_audio = [str(resolve_audio(row.get('audio') or row.get('audio_path') or '')) for row in rows if not resolve_audio(row.get('audio') or row.get('audio_path') or '').exists()]
    print(manifest.name, 'rows =', len(rows), 'counts =', dict(sorted(counts.items())), 'missing_audio =', len(missing_audio))
    if LIMIT == 0 and len(rows) != expected_rows:
        raise AssertionError(f'{manifest} 行数异常: {len(rows)} != {expected_rows}')
    if missing_audio:
        print('\n'.join(missing_audio[:20]))
        raise FileNotFoundError(f'{manifest} 有音频缺失: {len(missing_audio)}')

In [ ]:
# 可选 Hugging Face token。若模型下载遇到限流，在 Colab Secrets 设置 HF_TOKEN 后重跑本 cell。
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token detected from Colab Secrets.')
else:
    print('No HF token found in Colab Secrets. Public download will be used.')

In [ ]:
# 命令执行工具：打印 stdout/stderr tail，便于快速定位失败点。
def run_cmd(cmd, stderr_tail=16000, stdout_tail=12000):
    print('运行命令:')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True, capture_output=True)
    print('returncode =', result.returncode)
    if result.stdout:
        print('--- stdout tail ---')
        print(result.stdout[-stdout_tail:])
    if result.stderr:
        print('--- stderr tail ---')
        print(result.stderr[-stderr_tail:])
    result.check_returncode()
    return result

In [ ]:
# 生成 mini manifest：每个 split 取最多 7 条，优先覆盖 clean 和 degraded，先确认 base 推理通路。
MINI_DIR = OUTPUT_DIR / 'mini'
MINI_DIR.mkdir(parents=True, exist_ok=True)

def make_mini_manifest(source_manifest, output_path, limit):
    rows = read_jsonl(source_manifest)
    selected = []
    seen = set()
    for row in rows:
        scenario = row.get('scenario', '')
        if scenario not in seen:
            selected.append(row)
            seen.add(scenario)
        if len(selected) >= limit:
            break
    if len(selected) < min(limit, len(rows)):
        for row in rows:
            if row not in selected:
                selected.append(row)
            if len(selected) >= limit:
                break
    with output_path.open('w', encoding='utf-8') as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    return selected

MINI_TRAIN = MINI_DIR / 'v6a_train.mini.jsonl'
MINI_VAL = MINI_DIR / 'v6a_val.mini.jsonl'
mini_train_rows = make_mini_manifest(TRAIN_MANIFEST, MINI_TRAIN, MINI_LIMIT_PER_SPLIT)
mini_val_rows = make_mini_manifest(VAL_MANIFEST, MINI_VAL, MINI_LIMIT_PER_SPLIT)
print('mini train scenarios =', Counter(row.get('scenario', '') for row in mini_train_rows))
print('mini val scenarios =', Counter(row.get('scenario', '') for row in mini_val_rows))

In [ ]:
# 跑 mini base inference + WER，避免 full run 才发现环境或路径问题。
def base_infer(manifest, output_jsonl, limit=0):
    cmd = [
        sys.executable,
        'inference/qwen3_asr_base_infer.py',
        '--manifest', str(manifest),
        '--output-jsonl', str(output_jsonl),
        '--audio-root', str(AUDIO_ROOT),
        '--model-id', MODEL_ID,
        '--dtype', DTYPE,
        '--device-map', DEVICE_MAP,
        '--quantization', QUANTIZATION,
        '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--language', LANGUAGE,
    ]
    if limit > 0:
        cmd.extend(['--limit', str(limit)])
    run_cmd(cmd)

def eval_predictions(pred_jsonl, scored_jsonl, metrics_json, scenario_csv):
    cmd = [
        sys.executable,
        'evaluation/eval_wer.py',
        '--predictions-jsonl', str(pred_jsonl),
        '--scored-jsonl', str(scored_jsonl),
        '--metrics-json', str(metrics_json),
        '--metrics-by-scenario-csv', str(scenario_csv),
    ]
    run_cmd(cmd)

MINI_TRAIN_PRED = MINI_DIR / 'predictions.qwen3_asr_base_v6a_train.mini.jsonl'
MINI_TRAIN_SCORED = MINI_DIR / 'predictions.qwen3_asr_base_v6a_train.mini.scored.jsonl'
MINI_TRAIN_METRICS = MINI_DIR / 'metrics.qwen3_asr_base_v6a_train.mini.json'
MINI_TRAIN_CSV = MINI_DIR / 'metrics_by_scenario.qwen3_asr_base_v6a_train.mini.csv'

base_infer(MINI_TRAIN, MINI_TRAIN_PRED)
eval_predictions(MINI_TRAIN_PRED, MINI_TRAIN_SCORED, MINI_TRAIN_METRICS, MINI_TRAIN_CSV)
mini_scored = read_jsonl(MINI_TRAIN_SCORED)
print('mini scored rows =', len(mini_scored))
for row in mini_scored[:3]:
    print(row.get('scenario'), 'wer=', row.get('wer'), 'error=', row.get('error'))
    print('answer    :', row.get('answer'))
    print('prediction:', row.get('prediction'))

In [ ]:
# Full train/val base scoring。LIMIT=0 表示完整跑 1680/420；如果只是调试，可在配置里改 runtime.limit。
def run_split(split_name, manifest, expected_rows):
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    pred = split_dir / f'predictions.qwen3_asr_base_v6a_{split_name}.jsonl'
    scored = split_dir / f'predictions.qwen3_asr_base_v6a_{split_name}.scored.jsonl'
    metrics = split_dir / f'metrics.qwen3_asr_base_v6a_{split_name}.json'
    csv = split_dir / f'metrics_by_scenario.qwen3_asr_base_v6a_{split_name}.csv'
    analysis_dir = split_dir / 'error_analysis'
    difficulty = TRAIN_DIFFICULTY if split_name == 'train' else VAL_DIFFICULTY
    difficulty_summary = split_dir / f'difficulty_summary.{split_name}.json'
    difficulty_csv = split_dir / f'difficulty_by_scenario_bucket.{split_name}.csv'

    base_infer(manifest, pred, limit=LIMIT)
    eval_predictions(pred, scored, metrics, csv)
    run_cmd([
        sys.executable,
        'evaluation/analyze_errors.py',
        '--scored-jsonl', str(scored),
        '--output-dir', str(analysis_dir),
        '--top-k', str(int(config.get('analysis', {}).get('top_k', 40))),
    ])
    actual_expected = -1 if LIMIT > 0 else expected_rows
    run_cmd([
        sys.executable,
        'scripts/build_difficulty_manifest.py',
        '--scored-jsonl', str(scored),
        '--output-jsonl', str(difficulty),
        '--summary-json', str(difficulty_summary),
        '--summary-csv', str(difficulty_csv),
        '--split', split_name,
        '--expected-rows', str(actual_expected),
    ])
    return {
        'split': split_name,
        'predictions': pred,
        'scored': scored,
        'metrics': metrics,
        'scenario_csv': csv,
        'analysis_dir': analysis_dir,
        'difficulty': difficulty,
        'difficulty_summary': difficulty_summary,
        'difficulty_csv': difficulty_csv,
    }

train_outputs = run_split('train', TRAIN_MANIFEST, 1680)
val_outputs = run_split('val', VAL_MANIFEST, 420)

In [ ]:
# 汇总 train/val difficulty，用于决定 Notebook 12 的采样策略。
def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

train_summary = load_json(train_outputs['difficulty_summary'])
val_summary = load_json(val_outputs['difficulty_summary'])
train_rows = read_jsonl(TRAIN_DIFFICULTY)
val_rows = read_jsonl(VAL_DIFFICULTY)

if LIMIT == 0:
    assert len(train_rows) == 1680, len(train_rows)
    assert len(val_rows) == 420, len(val_rows)

required_fields = {'base_prediction', 'base_wer', 'difficulty_bucket', 'failure_tags'}
for label, rows in [('train', train_rows), ('val', val_rows)]:
    for idx, row in enumerate(rows[:20], start=1):
        missing = sorted(required_fields - set(row))
        if missing:
            raise AssertionError(f'{label} row {idx} missing {missing}')

combined = {
    'dataset': config['output'].get('dataset_name', 'v6a_hard_profile'),
    'model_id': MODEL_ID,
    'quantization': QUANTIZATION,
    'train': train_summary,
    'val': val_summary,
    'train_difficulty_manifest': str(TRAIN_DIFFICULTY.relative_to(PROJECT_DIR)),
    'val_difficulty_manifest': str(VAL_DIFFICULTY.relative_to(PROJECT_DIR)),
    'next_step': 'notebooks/12_train_lora_v6a_hard_profile_colab.ipynb',
}
SUMMARY_JSON.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_JSON.write_text(json.dumps(combined, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('v6A base difficulty ready')
print('train difficulty:', TRAIN_DIFFICULTY)
print('val difficulty:', VAL_DIFFICULTY)
print('summary:', SUMMARY_JSON)
print(json.dumps({
    'train_rows': len(train_rows),
    'val_rows': len(val_rows),
    'train_bucket_counts': train_summary.get('difficulty_bucket_counts'),
    'val_bucket_counts': val_summary.get('difficulty_bucket_counts'),
    'train_failure_tag_counts': train_summary.get('failure_tag_counts'),
    'val_failure_tag_counts': val_summary.get('failure_tag_counts'),
}, ensure_ascii=False, indent=2))